# Track 10 — Capstone: Code Review Assistant (PR diff 리뷰 + HITL merge 차단)

## 자동 코드 리뷰 + HITL 게이트란?

PR diff에 SQL 인젝션, `eval`/`exec`, 안전하지 않은 역직렬화가 들어오면 운영 사고로 이어질 수 있습니다. 이 캡스톤은 위험 탐지와 머지 차단을 세 단계로 나눕니다.

- **정규식 1차 필터:** 추가된(`+`) 라인에서 명백한 위험 패턴을 빠르게 찾습니다.
- **EXAONE 의미 리뷰:** diff 맥락을 읽고 심각도·이유·수정안을 구조화 JSON으로 받습니다.
- **HITL 머지 차단:** 되돌리기 어려운 조치는 사람 승인 뒤에만 실행합니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 정적 메트릭 데모 · 패키지 저장 헬퍼 | 공통 준비 |
| 2-1. 위험 탐지 | 정규식 스캔 + EXAONE 의미 리뷰 | 가벼운 탐지와 의미 판정 결합 |
| 2-2. HITL 게이트 | `block_merge` 승인/거부 경로와 anti-pattern | 고위험 조치를 사람 승인 뒤에 배치 |
| 3. 패키지 | findings·LLM 리뷰·HITL·정적 회귀 저장 | 제출·회귀 산출물 마감 |

## 이 노트북을 마치면

- PR diff 위험을 정규식, LLM 리뷰, 사람 승인으로 나눠 처리할 수 있습니다.
- EXAONE 리뷰 결과를 `answer` JSON으로 받고 `StructuredOutputPipeline`으로 검증할 수 있습니다.
- 승인 신호를 도구 인자가 아니라 클로저에 둬 위조를 막는 패턴을 이해할 수 있습니다.

**산출물:** `_out/03/capstone_package.json`  
**실행 조건:** 정규식 스캔·HITL·정적 메트릭은 API 키 없이 실행됩니다. EXAONE 의미 리뷰만 API 키가 필요하며, 키가 없으면 정규식 결과로 진행합니다.

> 요약: PR diff 위험을 찾고, 머지 차단을 사람 승인 게이트로 보호한 뒤 패키지로 마감하는 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로·클라이언트·골든 로더·정적 메트릭 데모·패키지 저장 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | HAS_API True/False` 한 줄이 출력됩니다.

**의미:** 이후 단계에서 쓸 경로·클라이언트가 맞는지 먼저 봅니다.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| HAS_API", HAS_API)

def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Static fixture metric demo (M1/M6/M9) over golden rows — metric MECHANICS, not live agent.
    # (kr) 정적 골든 fixture로 M1/M6/M9를 계산한다. 라이브 에이전트 성능이 아니라 메트릭 동작 예시다.
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {"n": len(rows), "M1_mean": mean(m1s), "M6_loose_mean": mean(m6s), "M9_mean": mean(m9s), "cases": cases}


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under TRACK10/_out/<nb>/.
    # (kr) TRACK10/_out/<nb>/capstone_package.json을 저장한다.
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path

**출력 해석:** `exaone 0.1.0 | HAS_API True/False` 한 줄이 보이면 준비 완료입니다.

- `HAS_API`가 이후 분기를 가릅니다: **`True`** 면 Session 2-1의 `llm_review`가 여기서 만든 `client` 로 EXAONE 의미 리뷰를 호출하고, **`False`** 면 그 단계만 `[SKIP]` 하고 정규식 결과로 폴백합니다 — 정규식 스캔·HITL·정적 메트릭은 두 경우 모두 동일하게 돕니다(키 없이도 완주).
- `TRACK10`·`DATA`가 **절대경로**로 잡혀, 이후 셀이 어느 CWD에서 실행돼도 같은 `pr_diff_sample.diff`·골든을 읽고 같은 `_out/03/` 위치에 저장합니다.

## Session 2. 위험 탐지 + HITL 게이트

### Session 2-1. 위험 탐지 — 정규식 1차 + EXAONE 의미 리뷰

**하는 일:** PR diff를 **두 층**으로 탐지합니다. ① `scan_diff` 정규식이 고정 위험 패턴을 **가볍게(API 키 불필요)** 1차로 걸러내고, ② `llm_review`가 **EXAONE을 리뷰어로** 불러 diff를 의미로 읽고 **심각도·이유·수정안**을 구조화 JSON으로 받습니다. 둘을 합쳐 "위험이 있나"를 판정합니다.

**왜 두 층인가:** 정규식은 `f"SELECT` 라는 *글자*만 보지, *"왜 위험한지"* 는 모릅니다. 그 판단(semantic judgment)이 **EXAONE이 맡는 역할**입니다. EXAONE은 추가 라인뿐 아니라 맥락(제거된 안전한 원본)까지 읽어 수정안을 제안합니다.

**키 게이트 + 폴백:** `llm_review`는 **`EXAONE_API_KEY`가 있을 때만** 호출됩니다. 키가 없으면 `[SKIP]` 후 정규식 결과만 사용하므로 노트북은 끝까지 실행됩니다. 영어 시스템 프롬프트 · 구조화 `answer` JSON · `StructuredOutputPipeline`(검증·복구) · 기본 샘플링을 따릅니다.

**정상(키 있을 때):** `[1] 정규식 스캔 … 위험 2건` 리포트에 이어 `[2] EXAONE 의미 리뷰 … 2건`(각 심각도/이유/수정안), 그리고 `→ 위험 발견: True …` 한 줄. (키 없으면 `[2]` 자리에 `[SKIP]`.)

**의미:** 가벼운 1차 필터 → 의미 기반 판정의 **2단 탐지**를 보여주고, 그 결과가 다음 HITL 게이트를 트리거합니다.

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable

DIFF = (DATA / "pr_diff_sample.diff").read_text(encoding="utf-8")
# (en) Map each risk pattern to a human-readable label so the scan reads like a review report.
# (kr) 각 위험 패턴을 사람이 읽을 라벨에 대응시켜, 스캔 결과가 리뷰 리포트처럼 보이게 한다.
RISK_PATTERNS = {
    "f\"SELECT": "SQL 인젝션 위험 (f-string 쿼리)",
    "eval(": "임의 코드 실행 (eval)",
    "exec(": "임의 코드 실행 (exec)",
    "pickle.loads": "안전하지 않은 역직렬화 (pickle)",
}


def scan_diff(text: str) -> list[dict[str, str]]:
    # (en) CHEAP first filter: fixed-string risk patterns on added (+) lines, no LLM needed.
    # (kr) 가벼운 1차 필터: 추가(+)된 라인의 고정 문자열 위험 패턴, LLM 불필요.
    findings = []
    for line in text.splitlines():
        if line.startswith("+") and not line.startswith("+++"):
            for pat, label in RISK_PATTERNS.items():
                if pat in line:
                    findings.append({"pattern": pat, "risk": label, "added_line": line[1:].strip()[:80]})
    return findings


# (en) EXAONE plays the REVIEWER: read the diff semantically and return severity/why/fix — the judgment a fixed-string scan cannot make (English system prompt + structured answer JSON).
# (kr) EXAONE이 '리뷰어' 역할: diff를 의미로 읽어 심각도·이유·수정안을 돌려준다 — 고정 문자열 스캔이 못 하는 '판단'(영어 시스템 프롬프트 + 구조화 answer JSON).
REVIEW_SYSTEM = (
    "You are a senior security-focused code reviewer. You are given a unified git diff. "
    "Review ONLY the risks introduced by added (+) lines. "
    "Respond with a single JSON object only, shaped as: "
    '{"answer": <one-line Korean summary in 높임말>, '
    '"findings": [{"severity": "high|medium|low", "line": <risky added code>, '
    '"risk": <short category>, "why": <why dangerous, Korean 높임말>, '
    '"fix": <concrete remediation, Korean 높임말>}]}'
)


def llm_review(diff_text: str) -> list[dict[str, Any]]:
    # (en) Best-effort semantic review: any failure (no key, endpoint down, bad/empty JSON, malformed findings) degrades to [] so the regex-only path keeps running.
    # (kr) 베스트-에포트 의미 리뷰: 어떤 실패(키 없음·엔드포인트 다운·JSON 깨짐/빈값·findings 형태 이상)든 []로 떨어져 정규식 단독 경로로 계속 돈다.
    if not HAS_API or client is None:
        print("  [SKIP] EXAONE 의미 리뷰는 EXAONE_API_KEY가 있어야 동작합니다 (정규식 결과만 사용).")
        return []
    messages = [
        exaone.llm.ExaoneMessage(role="system", content=REVIEW_SYSTEM),
        exaone.llm.ExaoneMessage(role="user", content=f"```diff\n{diff_text}\n```"),
    ]
    # (en) Keep sampling at defaults (temp=1.0/top_p=0.95/do_sample=True); thinking off for clean JSON.
    # (kr) 샘플링은 기본값을 유지한다(greedy 금지). 깔끔한 JSON을 위해 thinking만 끈다.
    options = exaone.llm.ExaoneGenerateOptions(enable_thinking=False, max_new_tokens=1200, response_format={"type": "json_object"})
    try:
        raw = client.chat(messages, options).content or ""
    except Exception as exc:
        print(f"  [WARN] EXAONE 호출 실패({type(exc).__name__}) — 정규식 결과만 사용.")
        return []
    # (en) Model output is non-deterministic — validate/repair the envelope (answer/findings) before trusting it.
    # (kr) 모델 출력은 비결정적이므로 외곽(answer/findings)을 검증·복구한 뒤 신뢰한다.
    parsed = exaone.output.StructuredOutputPipeline(required_keys=["answer", "findings"]).process(raw)
    if not parsed.success:
        print("  [WARN] 구조화 출력 검증 실패:", parsed.error, "— 정규식 결과만 사용.")
        return []
    # (en) Defensive: the pipeline checks only the envelope, so keep only dict-shaped findings.
    # (kr) 방어적 처리: 파이프라인은 외곽만 검증하므로 dict 형태 finding만 남긴다.
    items = parsed.data.get("findings", [])
    return [f for f in items if isinstance(f, dict)] if isinstance(items, list) else []


# (en) Layer 1 — cheap regex pre-filter (always runs, key-free).
# (kr) 1층 — 가벼운 정규식 1차 필터(항상 실행, 키 불필요).
findings = scan_diff(DIFF)
print(f"[1] 정규식 스캔 — 위험 {len(findings)}건 ({len(RISK_PATTERNS)}개 패턴):")
for i, f in enumerate(findings, 1):
    print(f"  {i}. [{f['risk']}]  {f['added_line']}")

# (en) Layer 2 — EXAONE semantic review (key-gated): severity + why + concrete fix.
# (kr) 2층 — EXAONE 의미 리뷰(키 게이트): 심각도 + 이유 + 구체적 수정안.
llm_findings = llm_review(DIFF)
print(f"\n[2] EXAONE 의미 리뷰 — {len(llm_findings)}건:")
for f in llm_findings:
    print(f"  [{f.get('severity')}] {f.get('risk')} — {str(f.get('line')).strip()[:60]}")
    print(f"     이유: {f.get('why')}")
    print(f"     수정: {f.get('fix')}")

# (en) Combine both layers; any hit means this PR carries risk → gate the merge in Session 2-2.
# (kr) 두 층을 합친다; 하나라도 걸리면 이 PR은 위험 → Session 2-2의 머지 게이트로.
risk_found = bool(findings) or bool(llm_findings)
print(f"\n→ 위험 발견: {risk_found} (정규식 {len(findings)}건 + EXAONE {len(llm_findings)}건) → HITL 머지 차단 게이트로")


**출력 해석:** 탐지가 두 층으로 돕니다 — `[1]` 정규식이 위험 **2건**을 라벨과 함께 잡고, `[2]` EXAONE이 같은 diff를 의미로 리뷰합니다.

- **`[1]` 정규식 (가벼운 1차):** 추가(`+`) 라인의 고정 패턴만 매칭 — `[SQL 인젝션 위험] query = f"SELECT..."`, `[임의 코드 실행] return eval(name, {}, ctx)`. 빠르고 키도 필요 없지만 *글자*만 봅니다.
- **`[2]` EXAONE 의미 리뷰 (판정):** 같은 2건을 잡되 **심각도(high)·이유·수정안**까지 냅니다. 핵심은 — EXAONE은 추가 라인뿐 아니라 **맥락(제거된 `-` 안전 원본)** 까지 읽어, 수정안으로 *파라미터화 쿼리*(`db.fetch_one("…id=%s", (user_id,))`)와 *`RULES[name]` 복원*을 제안합니다. 정규식이 절대 못 하는 **판단**이 이 역할에서 채워집니다. (모델 출력이라 표현은 실행마다 조금씩 달라지고, `StructuredOutputPipeline`이 `answer`·`findings` 키를 검증·복구합니다.)
- **키 게이트:** 위 `[2]`는 `EXAONE_API_KEY`가 있을 때만 나옵니다. 키가 없으면 `[2]` 자리에 `[SKIP]`이 찍히고 정규식 결과만으로 진행 — 노트북은 그대로 끝까지 실행됩니다(키 없이 완주 가능).
- **합산 판정:** `risk_found = (정규식 ∪ EXAONE)`가 `True` → 다음 Session 2-2의 HITL 머지 차단 게이트가 **이 탐지 결과 때문에** 트리거됩니다.

### Session 2-2. HITL 머지 차단 게이트

**하는 일:** 탐지가 위험을 찾았으므로(`risk_found`), 되돌리기 어려운 **머지 차단**을 사람 승인 게이트 뒤에 둔 `block_merge` 도구를 승인/거부 두 경로로 실행하고, 제어 플래그를 도구 인자로 흘리는 **anti-pattern**까지 보여줍니다.

**정상:** `block_merge` 두 경로의 **전체 `ToolResult`**(승인 `ok:true` / 거부 `error:"rejected_by_human"`)와 `anti-pattern …` 한 줄이 출력됩니다.

**의미:** 자동 탐지가 위험을 올리면, **고위험 조치(머지 차단)는 사람 승인 없이는 실행되지 않게** 묶는 3층 구조의 마지막 칸(사람)입니다.

In [ ]:
@dataclass
class HumanApprovalController:
    # (en) HITL gate: approves only when the human input is 'y'/'yes' (here a mock stdin).
    # (kr) HITL 게이트: 사람 입력이 'y'/'yes'일 때만 승인한다(여기서는 mock stdin).
    stdin: Callable[[], str]

    def request_approval(self, tool_name: str, args: dict[str, Any]) -> bool:
        return (self.stdin() or "").strip().lower() in ("y", "yes")


def make_block_merge(controller: HumanApprovalController) -> Callable[[str, dict[str, Any]], dict[str, Any]]:
    # (en) Bind the approval gate into the tool via closure so the schema stays clean (pr_id only).
    # (kr) 승인 게이트를 클로저로 도구에 묶어 스키마는 pr_id만 노출하게 깔끔히 유지한다.
    def _block_merge(_name: str, args: dict[str, Any]) -> dict[str, Any]:
        pr = args.get("pr_id", "")
        if not controller.request_approval("block_merge", {"pr_id": pr}):
            return exaone.tools.ToolResult.failure(source="block_merge", error="rejected_by_human").to_dict()
        return exaone.tools.ToolResult.success(content=f"merge blocked for {pr}", source="block_merge").to_dict()
    return _block_merge


BLOCK_SCHEMA = {
    "type": "function",
    "function": {
        "name": "block_merge",
        "description": "Block PR merge (requires human approval)",
        "parameters": {
            "type": "object",
            "required": ["pr_id"],
            "properties": {"pr_id": {"type": "string"}},
            "additionalProperties": False,
        },
    },
}


def block_merge_registry(controller: HumanApprovalController) -> exaone.tools.ToolRegistry:
    # (en) Fresh registry holding one approval-gated block_merge tool.
    # (kr) 승인 게이트가 걸린 block_merge 도구 하나를 담은 새 레지스트리.
    reg = exaone.tools.ToolRegistry()
    reg.register(exaone.tools.tool_from_callable("block_merge", BLOCK_SCHEMA, make_block_merge(controller)))
    return reg


# (en) Detection (Session 2-1) found risk → gate the irreversible merge-block behind human approval.
# (kr) 탐지(Session 2-1)가 위험을 찾음 → 되돌리기 어려운 머지 차단을 사람 승인 뒤에 둔다.
assert risk_found, "탐지가 위험을 찾았으므로 HITL 게이트가 필요합니다"
# (en) Approval path: human types 'y' → tool succeeds and the risky merge is blocked.
# (kr) 승인 경로: 사람이 'y' 입력 → 도구가 성공해 위험한 머지를 막는다.
approved = block_merge_registry(HumanApprovalController(stdin=lambda: "y")).execute("block_merge", {"pr_id": "PR-42"})
# (en) Rejection path: human declines → tool fails with rejected_by_human; nothing is blocked.
# (kr) 거부 경로: 사람이 거부 → 도구가 rejected_by_human으로 실패하고 아무것도 막지 않는다.
rejected = block_merge_registry(HumanApprovalController(stdin=lambda: "n")).execute("block_merge", {"pr_id": "PR-42"})

# (en) Show the FULL ToolResult of both paths so the ok/outcome/error contract is visible.
# (kr) 두 경로의 전체 ToolResult를 보여, ok/outcome/error 계약이 눈에 보이게 한다.
KEYS = ("ok", "outcome", "content", "error")
print("HITL block_merge — 두 경로의 ToolResult (거부는 outcome 이 아니라 error 로 판정):")
print("  승인('y') →", json.dumps({k: approved[k] for k in KEYS}, ensure_ascii=False))
print("  거부('n') →", json.dumps({k: rejected[k] for k in KEYS}, ensure_ascii=False))

# (en) Why bind via closure? A control flag passed as a tool ARG never reaches the callable — ToolRegistry.execute rejects it at schema validation and returns a tool_failure_payload.
# (kr) 왜 클로저로 묶나? 제어 플래그를 도구 인자로 흘리면 콜러블에 닿지도 못한다 — ToolRegistry.execute 가 스키마 검증에서 거부하고 tool_failure_payload 를 돌려준다.
leaked = block_merge_registry(HumanApprovalController(stdin=lambda: "y")).execute("block_merge", {"pr_id": "PR-42", "_auto": True})
print("\nanti-pattern — 제어 플래그(_auto)를 도구 인자로 흘리면 스키마가 막습니다:")
print("  ", leaked["error"])

hitl = {"approved": approved, "rejected": rejected}
# (en) Regression guard: approve→success, reject→rejected_by_human, smuggled flag→schema-rejected.
# (kr) 회귀 가드: 승인→성공, 거부→rejected_by_human, 도구 인자로 흘린 플래그→스키마 거부.
assert approved["ok"] is True and approved["content"] == "merge blocked for PR-42"
assert rejected["ok"] is False and rejected["error"] == "rejected_by_human"
assert leaked.get("_exaone_tool_failure") is True and "Additional properties" in leaked["error"]


**출력 해석:** `block_merge` HITL 게이트가 승인/거부 두 경로와 anti-pattern을 보여줍니다.

- **승인 경로:** `'y'` → `ok=true`, `outcome=success`, `content="merge blocked for PR-42"`.
- **거부 경로:** `'n'` → `ok=false`, `error="rejected_by_human"`. 모델이 단독으로 머지 차단을 실행하지 못합니다.
- **거부 판정:** `ToolResult.failure()`는 `outcome`을 `transport_error`로 남깁니다. 사람 거부는 `error == "rejected_by_human"`으로 읽습니다.
- **승인 위조 차단:** `_auto`를 도구 인자로 넣으면 `additionalProperties: false`가 거부합니다.
- **설계 이유:** HITL 제어는 도구 인자가 아니라 `make_block_merge` 클로저에 둡니다. 스키마는 `pr_id`만 노출합니다.


## Session 3. 패키지


### Session 3-1. 패키지

**하는 일:** diff findings·HITL 두 경로 데모·정적 회귀(M1/M6/M9)·트레이스·SLO를 한 JSON으로 묶어 저장하고, 산출물에 무엇이 담겼는지 요약합니다.

**정상:** `regression n=22 | M1=… M6=… M9(stub)=…`, `saved … _out/03/capstone_package.json`, `패키지 요약: {…}` 세 줄이 출력됩니다.

**의미:** 캡스톤 제출·회귀용 결과를 한 파일로 묶고, 그 안에 무엇이 들었는지 화면에서 바로 확인합니다.

In [ ]:
regression = regression_m1_m6_m9(load_capstone_golden("03"))
# (en) Surface the static metric-demo numbers inline (metric MECHANICS, not the agent's score).
# (kr) 정적 메트릭 데모 수치를 화면에 보여준다(에이전트 성능이 아니라 메트릭 동작 예시).
print(f"regression n={regression['n']} | M1={regression['M1_mean']:.2f} M6={regression['M6_loose_mean']:.2f} M9(stub)={regression['M9_mean']:.2f}")
session_trace = [
    {"event": "scan", "n_findings": len(findings)},
    {"event": "llm_review", "n_findings": len(llm_findings), "used_llm": HAS_API},
    {"event": "block_merge", "path": "approved", "ok": hitl["approved"]["ok"]},
    {"event": "block_merge", "path": "rejected", "ok": hitl["rejected"]["ok"], "error": hitl["rejected"]["error"]},
]
package_path = save_package("03", {"diff_findings": findings, "llm_review": llm_findings, "hitl_demo": hitl, "regression": regression, "session_trace": session_trace})
# (en) Recap what the submission artifact actually contains (otherwise only visible inside the JSON).
# (kr) 제출 산출물에 실제로 무엇이 담겼는지 요약한다(원래는 JSON 안에서만 보임).
print("패키지 요약:", json.dumps({
    "diff_findings": [f["risk"] for f in findings],
    "llm_review": len(llm_findings),
    "hitl": {"approved": approved["ok"], "rejected_by": rejected["error"]},
    "regression": {"n": regression["n"], "M1": round(regression["M1_mean"], 2)},
    "trace_events": [e["event"] for e in session_trace],
}, ensure_ascii=False))


**출력 해석:** `regression …`, `saved … _out/03/capstone_package.json`, `패키지 요약: {…}` 세 줄이 출력되면 이 캡스톤 패키지가 완성된 것입니다.

- `regression` 수치는 공통 골든 `all` 22행을 채점하는 **메트릭 동작 데모**일 뿐 에이전트 성능이 아닙니다(`03` 전용 행은 아직 없음): `M1=0.57` = `expected_answer`가 있는 7행 중 **4건 정확 일치**, `M6=0.50` = `required_keys`가 있는 6행 중 **3건이 키 충족**, `M9(stub)=0.53` = `grounding_context`가 있는 5행에 대한 `LengthRatioJudge` **길이비 근사**(충실도 측정이 아닌 테스트 전용 스텁).
- `패키지 요약` 한 줄은 저장된 JSON 안을 화면으로 끌어올린 것입니다 — `diff_findings`(정규식 라벨)·`llm_review`(EXAONE 의미 리뷰 건수, 키 없으면 0)·`hitl`(승인 `True` / 거부 `rejected_by_human`)·`regression`·`trace_events`(scan→llm_review→block_merge×2). 같은 내용이 `_out/03/capstone_package.json` 에 `slo`와 함께 담겨 제출·회귀 신호로 추적됩니다.
- `session_trace`의 `llm_review`이벤트는 `used_llm`(키 유무)을 함께 남겨, 이 산출물이 **정규식만으로** 나왔는지 **EXAONE 리뷰까지** 거쳤는지 구분됩니다.
- `_out/03/`는 절대경로로 잡힌 `TRACK10` 기준이라, 노트북을 어느 CWD 에서 실행해도 항상 같은 위치에 산출물이 저장됩니다.

## 마무리

이 캡스톤에서는 PR diff 위험을 **정규식 1차 + EXAONE 의미 리뷰**로 찾고, `block_merge`를 HITL 승인 게이트 뒤에 둔 뒤 `_out/03/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **탐지:** 정규식은 빠르게 위험 패턴을 찾고, EXAONE은 diff 맥락을 읽어 심각도·이유·수정안을 제안합니다.
- **폴백:** API 키가 없거나 LLM 호출이 실패하면 정규식 결과만으로 계속 진행합니다.
- **HITL:** 승인 시 `success`, 거부 시 `rejected_by_human`으로 갈립니다. 거부 판정은 `outcome`이 아니라 `error`를 봅니다.
- **스키마 보호:** `_auto` 같은 제어 플래그를 도구 인자로 넣으면 `additionalProperties: false`가 거부합니다. 승인 제어는 클로저에 둡니다.
- **패키지:** findings, LLM 리뷰, HITL 두 경로, 정적 메트릭, trace, SLO를 한 JSON으로 묶습니다.

**한계**
- 정규식은 고정 문자열 매칭이라 변형된 위험을 놓칠 수 있습니다.
- EXAONE 리뷰는 비결정적입니다. 운영에서는 골든셋, 심각도 합의 규칙, 리뷰 정책으로 안정화해야 합니다.
- HITL 입력은 데모용 mock입니다. 실제 운영에서는 인증, 감사 로그, 승인 UI가 필요합니다.
- M1/M6/M9는 정적 fixture를 채점하는 메트릭 데모이며 에이전트 성능이 아닙니다.

**다음:** `10_04` Data Analyst. 운영화는 `10_07` 프로덕션 하네스를 참고하세요.

## 체크포인트

- [ ] Session 2-1 정규식 스캔 + EXAONE 의미 리뷰 확인
- [ ] Session 2-2 `block_merge` 승인/거부와 스키마 거부 확인
- [ ] Session 3 `_out/03/capstone_package.json` 저장 + `패키지 요약` 출력
